This project requires Python 3.7 or above:

In [1]:
import sys
print(sys.executable)

assert sys.version_info >= (3, 7)

/usr/bin/python3


In [2]:
# Is this notebook running on Colab or Kaggle?
IS_COLAB = "google.colab" in sys.modules
IS_KAGGLE = "kaggle_secrets" in sys.modules

if IS_COLAB or IS_KAGGLE:
    %pip uninstall -y gym  # to avoid any conflicts with Gymnasium
    %pip install -q box2d

Found existing installation: gym 0.25.2
Uninstalling gym-0.25.2:
  Successfully uninstalled gym-0.25.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 48.4 MB/s eta 0:00:00


In [3]:
from packaging import version
import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

In [4]:
!/usr/bin/python3 -m pip install --user gymnasium[atari] autorom
!/usr/bin/python3 -m pip install -U ale-py
!/usr/bin/python3 -m pip install -U stable-baselines3[extra]



  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 6.5 MB/s eta 0:00:00


In [5]:
!/usr/bin/python3 -m pip show autorom
!/usr/bin/python3 -m autorom.accept_rom_license


Name: AutoROM
Version: 0.6.1
Summary: Automated installation of Atari ROMs for Gym/ALE-Py
Home-page: https://github.com/Farama-Foundation/AutoROM
Author: Farama Foundation
Author-email: contact@farama.org
License: MIT
Location: /root/.local/lib/python3.12/site-packages
Requires: click, requests
Required-by: 
/usr/bin/python3: Error while finding module specification for 'autorom.accept_rom_license' (ModuleNotFoundError: No module named 'autorom')


In [6]:
import gymnasium as gym
import ale_py

# Create the Pong environment
env = gym.make("ALE/Pong-v5", render_mode="human")

# Reset the environment to get the initial observation
observation, info = env.reset()

# Print the shape of the observation space
print(f"Observation space shape: {env.observation_space.shape}")

# Print the number of possible actions
print(f"Number of actions: {env.action_space.n}")

Observation space shape: (210, 160, 3)
Number of actions: 6


In [7]:
!/usr/bin/python3 -m pip install -U pip setuptools wheel
!/usr/bin/python3 -m pip  install -U "stable-baselines3[extra]"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 23.6 MB/s eta 0:00:00
  Attempting uninstall: setuptools
    Found existing installation: setuptools 75.2.0
    Uninstalling setuptools-75.2.0:
      Successfully uninstalled setuptools-75.2.0
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [8]:
%pip install -U stable-baselines3[extra]

In [9]:

import site, sys, os

user_site = site.getusersitepackages()
print("User site:", user_site)
print("In sys.path:", user_site in sys.path)

if user_site not in sys.path:
    sys.path.append(user_site)
    print("Added user site to sys.path")

# Now try imports
from stable_baselines3 import DQN
import gymnasium as gym
print("Imports OK after sys.path fix")


User site: /root/.local/lib/python3.12/site-packages
In sys.path: False
Added user site to sys.path
Imports OK after sys.path fix


In [10]:

import sys, subprocess, pkgutil

print("Python:", sys.executable)
print("stable_baselines3 loader:", pkgutil.find_loader("stable_baselines3"))

print("pip install path:")
subprocess.run([sys.executable, "-m", "pip", "show", "stable-baselines3"])


Python: /usr/bin/python3
stable_baselines3 loader: <_frozen_importlib_external.SourceFileLoader object at 0x7dfe31fcc200>
pip install path:


/tmp/ipykernel_331/1056075604.py:4: DeprecationWarning: 'pkgutil.find_loader' is deprecated and slated for removal in Python 3.14; use importlib.util.find_spec() instead
  print("stable_baselines3 loader:", pkgutil.find_loader("stable_baselines3"))


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'show', 'stable-baselines3'], returncode=0)

In [11]:

# (Optional) Confirm imports
import gymnasium as gym
from stable_baselines3 import DQN
print("Gymnasium:", gym.__version__)
print("SB3 DQN imported OK")


Gymnasium: 1.2.3
SB3 DQN imported OK


In [14]:
import gymnasium as gym
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack

# ---- Create a vectorized Atari env with standard preprocessing ----
# - make_atari_env applies the usual Atari wrappers (frame-skip, grayscale+resize)
# - VecFrameStack stacks 4 frames for the DQN CNN policy
env_id = "ALE/Pong-v5"
train_env = make_atari_env(env_id, n_envs=4, seed=0)  # 4 parallel envs for faster training
train_env = VecFrameStack(train_env, n_stack=4)

# ---- DQN hyperparameters known to work on Atari ----
model = DQN(
    "CnnPolicy",
    train_env,
    learning_rate=1e-4,
    buffer_size=100_000,
    learning_starts=10_000,
    batch_size=32,
    gamma=0.99,
    train_freq=4,                 # update every 4 steps
    target_update_interval=1_000, # target network update
    exploration_fraction=0.1,
    exploration_final_eps=0.01,
    verbose=1,
    tensorboard_log="./tb_logs",  # optional
)

# ---- Train ----
# For a quick demo, 200k steps. For stronger play: 500k–2M steps.
total_timesteps = 1200_000
model.learn(total_timesteps=total_timesteps)

# ---- Save the model ----
model.save("pong_dqn")
train_env.close()

Le flux de sortie a été tronqué et ne contient que les 5000 dernières lignes.
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.00494  |
|    n_updates        | 56890    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 822      |
|    ep_rew_mean      | -20.9    |
|    exploration_rate | 0.01     |
| time/               |          |
|    episodes         | 4744     |
|    fps              | 87       |
|    time_elapsed     | 10510    |
|    total_timesteps  | 921132   |
| train/              |          |
|    learning_rate    | 0.0001   |
|    loss             | 0.00581  |
|    n_updates        | 56945    |
----------------------------------
----------------------------------
| rollout/            |          |
|    ep_len_mean      | 820      |
|    ep_rew_mean      | -20.9    |
|    exploration_rate | 0.01     |
| time/               |          |
|    episode

In [18]:
import os, glob, numpy as np
from stable_baselines3.common.vec_env import VecVideoRecorder
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3 import DQN

env_id = "ALE/Pong-v5"
video_dir = "./videos"
os.makedirs(video_dir, exist_ok=True)

eval_env = make_atari_env(env_id, n_envs=1, seed=123)
eval_env = VecFrameStack(eval_env, n_stack=4)

video_length = 5_000
eval_env = VecVideoRecorder(
    eval_env,
    video_folder=video_dir,
    record_video_trigger=lambda step: step == 0,
    video_length=video_length,
    name_prefix="dqn-pong",
)

model = DQN.load("pong_dqn")

num_eval_episodes = 20
for ep in range(1, num_eval_episodes + 1):
    obs = eval_env.reset()
    done = False
    total_reward = 0.0
    frame_count = 0

    while not done and frame_count < video_length:
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = eval_env.step(action)
        frame_count += 1
        total_reward += float(rewards[0])
        done = bool(dones[0])

    print(f"Episode {ep:03d} | Frames: {frame_count} | Return: {total_reward:.1f}")

eval_env.close()

# Show most recent video
latest = sorted(glob.glob(os.path.join(video_dir, "*.mp4")))[-1]
from IPython.display import Video
Video(latest, embed=True)


Episode 001 | Frames: 238 | Return: -21.0
Episode 002 | Frames: 205 | Return: -21.0
Episode 003 | Frames: 287 | Return: -21.0
Episode 004 | Frames: 224 | Return: -20.0
Episode 005 | Frames: 192 | Return: -21.0
Episode 006 | Frames: 221 | Return: -20.0
Episode 007 | Frames: 202 | Return: -21.0
Episode 008 | Frames: 220 | Return: -21.0
Episode 009 | Frames: 196 | Return: -21.0
Episode 010 | Frames: 185 | Return: -21.0
Episode 011 | Frames: 181 | Return: -21.0
Episode 012 | Frames: 229 | Return: -20.0
Episode 013 | Frames: 226 | Return: -20.0
Episode 014 | Frames: 237 | Return: -21.0
Episode 015 | Frames: 222 | Return: -21.0
Episode 016 | Frames: 183 | Return: -21.0
Episode 017 | Frames: 202 | Return: -21.0
Episode 018 | Frames: 251 | Return: -20.0
Episode 019 | Frames: 202 | Return: -21.0
Episode 020 | Frames: 179 | Return: -21.0
Moviepy - Building video /content/videos/dqn-pong-step-0-to-step-5000.mp4.
Moviepy - Writing video /content/videos/dqn-pong-step-0-to-step-5000.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/dqn-pong-step-0-to-step-5000.mp4


In [17]:

from google.colab import files
files.download("./videos/rl-video-episode-18.mp4")


FileNotFoundError: Cannot find file: ./videos/rl-video-episode-18.mp4

In [ ]:

from IPython.display import Video
Video("./videos/rl-video-episode-99.mp4")


In [20]:
import os
import glob
import numpy as np
from IPython.display import Video, display

from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_atari_env
from stable_baselines3.common.vec_env import VecFrameStack
from stable_baselines3.common.vec_env import VecVideoRecorder

# --- Paramètres ---
env_id = "ALE/Pong-v5"
video_dir = "./videos"
os.makedirs(video_dir, exist_ok=True)

# 1) Charger le modèle
model = DQN.load("pong_dqn")

# 2) Créer l'env d'évaluation (n_envs=1) + preprocessing Atari standard
eval_env = make_atari_env(env_id, n_envs=1, seed=123)
eval_env = VecFrameStack(eval_env, n_stack=4)

# 3) Enregistrer CHAQUE épisode (record_video_trigger=True à chaque reset)
video_length = 5_000  # cap en nombre de steps pour la vidéo
eval_env = VecVideoRecorder(
    eval_env,
    video_folder=video_dir,
    record_video_trigger=lambda step: True,   # <--- filmer chaque épisode
    video_length=video_length,
    name_prefix="dqn-pong",
)

# 4) Exécuter N épisodes, stocker les rewards totaux
num_eval_episodes = 50
episode_returns = []

for ep in range(1, num_eval_episodes + 1):
    obs = eval_env.reset()
    done = False
    total_reward = 0.0
    frame_count = 0

    while not done and frame_count < video_length:
        action, _ = model.predict(obs, deterministic=True)
        obs, rewards, dones, infos = eval_env.step(action)
        frame_count += 1
        total_reward += float(rewards[0])
        done = bool(dones[0])

    episode_returns.append(total_reward)
    print(f"Episode {ep:03d} | Frames: {frame_count} | Return: {total_reward:.1f}")

# 5) Fermer pour flusher les vidéos
eval_env.close()

# 6) Récupérer la liste des vidéos et les trier dans l'ordre de création
video_files = sorted(glob.glob(os.path.join(video_dir, "*.mp4")), key=os.path.getmtime)

# Sanity-check: on s'attend à 1 fichier par épisode
if len(video_files) != len(episode_returns):
    print(f"[WARN] Mismatch: {len(video_files)} fichiers vidéo pour {len(episode_returns)} épisodes.")
    # On continue quand même en prenant le min commun
    min_len = min(len(video_files), len(episode_returns))
    video_files = video_files[:min_len]
    episode_returns = episode_returns[:min_len]

# 7) Trouver le meilleur épisode
best_idx = int(np.argmax(episode_returns))
best_reward = episode_returns[best_idx]
best_video = video_files[best_idx]

print(f"\nBest episode: #{best_idx+1} with return={best_reward:.1f}")
print(f"Best video file: {best_video}")

# 8) (Optionnel) Nettoyer les autres vidéos et ne garder que la meilleure
#    Décommentez si vous voulez ne conserver que la meilleure
# for i, f in enumerate(video_files):
#     if i != best_idx:
#         try:
#             os.remove(f)
#         except OSError:
#             pass

# 9) Afficher la meilleure vidéo dans le notebook
display(Video(best_video, embed=True))

Episode 001 | Frames: 238 | Return: -21.0
Moviepy - Building video /content/videos/dqn-pong-step-238-to-step-5238.mp4.
Moviepy - Writing video /content/videos/dqn-pong-step-238-to-step-5238.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/dqn-pong-step-238-to-step-5238.mp4
Episode 002 | Frames: 205 | Return: -21.0
Moviepy - Building video /content/videos/dqn-pong-step-443-to-step-5443.mp4.
Moviepy - Writing video /content/videos/dqn-pong-step-443-to-step-5443.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/dqn-pong-step-443-to-step-5443.mp4
Episode 003 | Frames: 287 | Return: -21.0
Moviepy - Building video /content/videos/dqn-pong-step-730-to-step-5730.mp4.
Moviepy - Writing video /content/videos/dqn-pong-step-730-to-step-5730.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/dqn-pong-step-730-to-step-5730.mp4
Episode 004 | Frames: 224 | Return: -20.0
Moviepy - Building video /content/videos/dqn-pong-step-954-to-step-5954.mp4.
Moviepy - Writing video /content/videos/dqn-pong-step-954-to-step-5954.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/dqn-pong-step-954-to-step-5954.mp4
Episode 005 | Frames: 192 | Return: -21.0
Moviepy - Building video /content/videos/dqn-pong-step-1146-to-step-6146.mp4.
Moviepy - Writing video /content/videos/dqn-pong-step-1146-to-step-6146.mp4



...

Moviepy - Done !
Moviepy - video ready /content/videos/dqn-pong-step-9663-to-step-14663.mp4
Episode 047 | Frames: 206 | Return: -21.0
Moviepy - Building video /content/videos/dqn-pong-step-9869-to-step-14869.mp4.
Moviepy - Writing video /content/videos/dqn-pong-step-9869-to-step-14869.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/dqn-pong-step-9869-to-step-14869.mp4
Episode 048 | Frames: 190 | Return: -21.0
Moviepy - Building video /content/videos/dqn-pong-step-10059-to-step-15059.mp4.
Moviepy - Writing video /content/videos/dqn-pong-step-10059-to-step-15059.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/dqn-pong-step-10059-to-step-15059.mp4
Episode 049 | Frames: 220 | Return: -20.0
Moviepy - Building video /content/videos/dqn-pong-step-10279-to-step-15279.mp4.
Moviepy - Writing video /content/videos/dqn-pong-step-10279-to-step-15279.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/dqn-pong-step-10279-to-step-15279.mp4
Episode 050 | Frames: 180 | Return: -21.0
Moviepy - Building video /content/videos/dqn-pong-step-10279-to-step-15279.mp4.
Moviepy - Writing video /content/videos/dqn-pong-step-10279-to-step-15279.mp4



Moviepy - Done !
Moviepy - video ready /content/videos/dqn-pong-step-10279-to-step-15279.mp4

Best episode: #4 with return=-20.0
Best video file: ./videos/dqn-pong-step-730-to-step-5730.mp4
